# Context editing

Unlike summarization, this middleware clears old tool results rather than asking an LLM to rewrite the conversation. Compare tool-message lengths before and in the model request; recent results remain available.

[Context editing](https://docs.langchain.com/oss/python/langchain/middleware/built-in#context-editing)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt, tool_conversation
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from langchain.agents import create_agent
from langchain.agents.middleware import ClearToolUsesEdit, ContextEditingMiddleware
from model_config import get_model, run_config
from copy import deepcopy
from lab.faults import ScriptedModel
from langchain_core.messages import AIMessage


In [ ]:
# Execute
history = tool_conversation()
print("BEFORE:", [(m.tool_call_id, len(m.content)) for m in history if m.type == "tool"])
trace = TraceMiddleware("15_context", pretty=True)
agent = create_agent(
    model=get_model(),
    middleware=[
        ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=300, keep=1)]),
        trace,
    ],
)
result = agent.invoke({"messages": history}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(result["messages"], pretty=True)
request = next(event["payload"] for event in trace.events if event["kind"] == "MODEL REQUEST")
print("MODEL SEES:", [(m.tool_call_id, len(m.content)) for m in request["messages"] if m.type == "tool"])

### Protect tools and clear old call inputs

`keep` protects the most recent results; `exclude_tools` protects selected tools regardless of age. `clear_tool_inputs` removes the arguments of cleared calls as well as their outputs. `placeholder` replaces cleared results, and `clear_at_least` sets a minimum reclamation target. Token counting can use approximate counts or the model counter.

In [ ]:
# Execute
edited_history = deepcopy(tool_conversation())
protected_tool = "protected_handbook"
first_call = next(m.tool_calls[0] for m in edited_history if m.type == "ai" and m.tool_calls)
first_call["name"] = protected_tool
next(m for m in edited_history if m.type == "tool" and m.tool_call_id == first_call["id"]).name = protected_tool
protected_trace = TraceMiddleware("15_protected", pretty=True)
protected_agent = create_agent(model=ScriptedModel(responses=[AIMessage(content="The latest and protected tool results remain available.")]), middleware=[
    ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=300, keep=1, clear_at_least=0, clear_tool_inputs=True, exclude_tools=[protected_tool], placeholder="[OLD RESULT CLEARED]")]),
    protected_trace,
])
protected_result = protected_agent.invoke({"messages": edited_history}, config=run_config())

In [ ]:
# Inspect output
show_messages(protected_result["messages"], pretty=True)
protected_request = next(e["payload"]["messages"] for e in protected_trace.events if e["kind"] == "MODEL REQUEST")
print("Protected tool:", protected_tool)
print("Tool result previews:", [(m.name, m.content[:55]) for m in protected_request if m.type == "tool"])
assert all(m.content != "[OLD RESULT CLEARED]" for m in protected_request if m.type == "tool" and m.name == protected_tool)
assert any(m.type == "tool" and m.content == "[OLD RESULT CLEARED]" for m in protected_request)
assert any(m.type == "ai" and any(call["args"] == {} for call in m.tool_calls) for m in protected_request)